In [1]:
import boto3
import zipfile
import os

BUCKET_NAME = 'projekt-ml-odpady' 
FILE_NAME = 'archive.zip'

s3 = boto3.client('s3')

print("Pobieranie pliku z S3...")
s3.download_file(BUCKET_NAME, FILE_NAME, FILE_NAME)

print("Rozpakowywanie... (to może potrwać chwilę)")
with zipfile.ZipFile(FILE_NAME, 'r') as zip_ref:
    zip_ref.extractall('data')

print("Gotowe! Dane są w folderze 'data'.")

Pobieranie pliku z S3...
Rozpakowywanie... (to może potrwać chwilę)
Gotowe! Dane są w folderze 'data'.


In [2]:
%pip install ipywidgets gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 38.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 79.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ipywidgets]3 [ipywidgets]
Note: you may need to restart the kernel to use updated packages.


In [5]:
%pip install gradio

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached markupsafe-3.0.3-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.7 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached filelock-3.29.4-py3-none-any.whl.metadata (2.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 151.8 MB/s  0:00:00m0:00:01
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 54.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 160.8 MB/s  0:00:00
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Using cached markupsafe-3.0.3-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (20 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 142.9 MB/s  0:00:00
Using cached filelock-3.2

In [3]:
%pip install tensorflow matplotlib

  Using cached matplotlib-3.10.9-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (52 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached ml_dtypes-0.5.4-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.9 kB)
  Using cached contourpy-1.3.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.63.0-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (118 kB)
  Using cached kiwisolver-1.5.0-cp310-cp310-manylinux_2_12_x86_64.manylinux2010_x86_64.whl.metadata (5.1 kB)
  Using cached pillow-12.2.0-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.2/572.2 MB 55.0 MB/s  0:00:07m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 M

In [7]:
import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory
import matplotlib.pyplot as plt

# 1. Przygotowanie danych
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
DATA_DIR = 'data/garbage classification/Garbage classification' # sprawdź czy taka ścieżka się stworzyła

train_dataset = image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

validation_dataset = image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_dataset.class_names
print(f"Klasy: {class_names}")

# 2. Budowa modelu (Transfer Learning)
base_model = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3),
                                               include_top=False,
                                               weights='imagenet')
base_model.trainable = False # Zamrażamy bazę

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(len(class_names), activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# 3. Trening
print("Start treningu...")
history = model.fit(train_dataset, validation_data=validation_dataset, epochs=5)

# 4. Wykresy (dla kolegów do raportu!)
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.legend()
plt.title('Dokładność modelu')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.title('Strata modelu')
plt.show()

Found 2527 files belonging to 6 classes.
Using 2022 files for training.
Found 2527 files belonging to 6 classes.
Using 505 files for validation.
Klasy: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
Start treningu...
Epoch 1/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 98s 1s/step - accuracy: 0.4238 - loss: 1.4980 - val_accuracy: 0.5426 - val_loss: 1.1494
Epoch 2/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 88s 1s/step - accuracy: 0.6019 - loss: 1.0694 - val_accuracy: 0.6238 - val_loss: 1.0121
Epoch 3/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 93s 1s/step - accuracy: 0.6627 - loss: 0.9554 - val_accuracy: 0.6317 - val_loss: 0.9590
Epoch 4/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 138s 1s/step - accuracy: 0.6845 - loss: 0.8773 - val_accuracy: 0.6673 - val_loss: 0.9144
Epoch 5/5
64/64 ━━━━━━━━━━━━━━━━━━━━ 144s 1s/step - accuracy: 0.7132 - loss: 0.8248 - val_accuracy: 0.6673 - val_loss: 0.8960


<Figure size 1000x400 with 2 Axes>

In [10]:
import gradio as gr
import numpy as np
from PIL import Image
import tensorflow as tf

def predict_waste(img):
    # Konwersja i zmiana rozmiaru
    img = Image.fromarray(img.astype('uint8'), 'RGB')
    img = img.resize((224, 224))
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0

    # Predykcja
    prediction = model.predict(img_array)[0]
    
    # Przygotowanie wyników
    results = {class_names[i]: float(prediction[i]) for i in range(len(class_names))}
    return results

# Interfejs
demo = gr.Interface(
    fn=predict_waste, 
    inputs=gr.Image(sources=["webcam"]), 
    outputs=gr.Label(num_top_classes=3),
    title="Klasyfikacja Odpadów - LIVE"
)

# share=True stworzy publiczny link, który na 100% zadziała
demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://13e9bf87995b8cfeb9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step


In [8]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.legend()
plt.title('Dokładność modelu')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.title('Strata modelu')
plt.show()

<Figure size 1000x400 with 2 Axes>

In [9]:
model.save('model_odpady_final.h5')
print("Model zapisany! Możesz go teraz pobrać z panelu po lewej (odśwież listę plików).")

Model zapisany! Możesz go teraz pobrać z panelu po lewej (odśwież listę plików).
